In [1]:
import time
import os

import porespy as ps
import numpy as np
import scipy as sc
import pypardiso

os.chdir("..")
%run .\pyflowsolver\volumeManager.py
%run .\pyflowsolver\sparseArray.py
%run .\pyflowsolver\fastLaplacian.py
%run .\pyflowsolver\darcySolver.py
os.chdir("notebooks")

In [2]:
SIZE = 3
vol = ps.generators.blobs(shape=(SIZE, SIZE, SIZE), blobiness=0.4, porosity=0.55)
vol, n_lab = sc.ndimage.label(vol)
vol = (vol==1)
if vol.sum() == 0:
    print('error')
else:
    print('image OK')

cond_vol = (vol==1)*100 #porosity map ndarray uint8 0..100
cond_vol = fast_laplacian_volume_generator(
    cond_vol, 
    (1., 1., 1.), 
    )

#cond_vol[:cond_vol.shape[0]//2, :, :] *= 0.0000001
cond_vol[:cond_vol.shape[0]//2, :, :] *= 0.00001

volume_manager = VolumeManager(cond_vol)

image OK


In [3]:
if vol.shape[0] <= 30: # 30 for a 64 Gb RAM system
    dense_A, dense_b = volume_manager.get_linear_system()
    solution_template = np.linalg.solve(dense_A, dense_b)
    raveled_template = volume_manager.ravel_dense_solution(solution_template)
else:
    raveled_template = None

In [4]:
solver = DarcySolver()
sparse_A, sparse_b = volume_manager.get_sparse_system_jit()

In [5]:
if sparse_b.size < 150000:
    start_time = time.perf_counter()
    solution, error, iterations = solver.solve_jit(
        sparse_A, 
        sparse_b, 
        parallel=12, 
        max_iterations=100000, 
        target_error=1e-6,
    )
    run_time = time.perf_counter() - start_time
    if raveled_template is not None:
        raveled_solution = volume_manager.ravel_sparse_solution(solution)
        diff = np.abs(raveled_solution - raveled_template)
        print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
    else:
        print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

Error: [9.596133e-07]   Iterations: 42   Mean Error: 0.0016655971994623542   Max error: 0.008530199527740479   Run time: 4.02690789999906


In [6]:
start_time = time.perf_counter()
max_iterations = sparse_b.size
solution, error, iterations = solver._solve_cg(
        sparse_A.val,
        sparse_A.col_idx,
        sparse_A.row_ptr,
        sparse_b,
        max_iterations=max_iterations*100, # sqrt(n) for n x n system
        target_error=1.0e-9, # 1.0e-6
        X0=np.zeros(sparse_b.size, dtype=np.float64),
        threads=1,
    )
run_time = time.perf_counter() - start_time
if raveled_template is not None:
    raveled_solution = volume_manager.ravel_sparse_solution(solution)
    diff = np.abs(raveled_solution - raveled_template)
    print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
else:
    print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

Error: 2.6904229396296624e-12   Iterations: 30   Mean Error: 2.759474315716659e-10   Max error: 7.450580596923828e-09   Run time: 3.8601981999818236


In [7]:
P_val, P_col_idx, P_row_ptr = _get_diagonal_preconditioner(
    A_val = sparse_A.val, 
    A_col_idx=sparse_A.col_idx, 
    A_row_ptr=sparse_A.row_ptr, 
    threads=1,
    )
print(P_val)
print(P_col_idx)
print(P_row_ptr)

start_time = time.perf_counter()
max_iterations = sparse_b.size
solution, error, iterations = solver._solve_pcg(
        sparse_A.val,
        sparse_A.col_idx,
        sparse_A.row_ptr,
        P_val, 
        P_col_idx, 
        P_row_ptr,
        sparse_b,
        max_iterations=max_iterations*100, # sqrt(n) for n x n system
        target_error=1.0e-9, # 1.0e-6
        X0=np.zeros(sparse_b.size, dtype=np.float64),
        threads=1,
    )
run_time = time.perf_counter() - start_time
if raveled_template is not None:
    raveled_solution = volume_manager.ravel_sparse_solution(solution)
    diff = np.abs(raveled_solution - raveled_template)
    print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
else:
    print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

[-8.33336841e+04 -1.01011336e+04 -1.00000503e+05 -5.88237046e+04
 -8.00000000e-01 -1.13009537e+00 -3.53826239e-01 -8.00000000e-01
 -1.33332444e+00 -8.81149597e-01 -1.00000000e+00 -1.33333333e+00
 -1.13010175e+00 -1.00000000e+00 -2.00000000e+00 -1.33333333e+00]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
Error: 2.647893889429242e-10   Iterations: 13   Mean Error: 3.104408463627806e-08   Max error: 5.066394805908203e-07   Run time: 1.7170393000124022


In [8]:
from pypardiso import spsolve
from scipy.sparse import csc_matrix, csr_matrix


In [9]:
A = csr_matrix( (sparse_A.val, sparse_A.col_idx, np.append(sparse_A.row_ptr,sparse_A.val.size)) )

In [10]:
spsolve(A, sparse_b)

array([0.25060651, 0.05033084, 0.31501401, 0.09679374, 0.83772067,
       0.39865234, 0.07598852, 0.84404412, 0.4563293 , 0.12629426,
       0.92617064, 0.94590689, 0.03305114, 0.86063842, 0.51638305,
       0.17212768])

In [11]:
start_time = time.perf_counter()
max_iterations = sparse_b.size
A = csr_matrix( (sparse_A.val, sparse_A.col_idx, np.append(sparse_A.row_ptr,sparse_A.val.size)) )
solution = spsolve(A, sparse_b)
error = 0
iterations = 0
run_time = time.perf_counter() - start_time
if raveled_template is not None:
    raveled_solution = volume_manager.ravel_sparse_solution(solution)
    diff = np.abs(raveled_solution - raveled_template)
    print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
else:
    print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

Error: 0   Iterations: 0   Mean Error: 2.759474315716659e-10   Max error: 7.450580596923828e-09   Run time: 0.000526399991940707


In [12]:
solution

array([0.25060651, 0.05033084, 0.31501401, 0.09679374, 0.83772067,
       0.39865234, 0.07598852, 0.84404412, 0.4563293 , 0.12629426,
       0.92617064, 0.94590689, 0.03305114, 0.86063842, 0.51638305,
       0.17212768])

In [13]:
SIZE = 5
vol = np.zeros((SIZE,)*3, dtype=np.uint8)
vol[1:-1, 1:-1, :] = 1

cond_vol = vol*100 #porosity map ndarray uint8 0..100
cond_vol = fast_laplacian_volume_generator(
    cond_vol, 
    (1., 1., 1.), 
    )

#cond_vol[:cond_vol.shape[0]//2, :, :] *= 0.0000001
#cond_vol[:cond_vol.shape[0]//2, :, :] *= 0.00001

volume_manager = VolumeManager(cond_vol)

solver = DarcySolver()
sparse_A, sparse_b = volume_manager.get_sparse_system_jit()

In [21]:
start_time = time.perf_counter()
max_iterations = sparse_b.size
A = csr_matrix( (sparse_A.val, sparse_A.col_idx, np.append(sparse_A.row_ptr,sparse_A.val.size)) )
solution = spsolve(A, sparse_b)
error = 0
iterations = 0
run_time = time.perf_counter() - start_time
print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")
raveled_solution = volume_manager.ravel_sparse_solution(solution)

Error: 0   Iterations: 0   Run time: 0.00046660000225529075


In [22]:
raveled_solution

array([[[0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ]],

       [[0. , 0. , 0. , 0. , 0. ],
        [0.9, 0.7, 0.5, 0.3, 0.1],
        [0.9, 0.7, 0.5, 0.3, 0.1],
        [0.9, 0.7, 0.5, 0.3, 0.1],
        [0. , 0. , 0. , 0. , 0. ]],

       [[0. , 0. , 0. , 0. , 0. ],
        [0.9, 0.7, 0.5, 0.3, 0.1],
        [0.9, 0.7, 0.5, 0.3, 0.1],
        [0.9, 0.7, 0.5, 0.3, 0.1],
        [0. , 0. , 0. , 0. , 0. ]],

       [[0. , 0. , 0. , 0. , 0. ],
        [0.9, 0.7, 0.5, 0.3, 0.1],
        [0.9, 0.7, 0.5, 0.3, 0.1],
        [0.9, 0.7, 0.5, 0.3, 0.1],
        [0. , 0. , 0. , 0. , 0. ]],

       [[0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. ]]], dtype=float32)

In [30]:
scale = (1., 1., 1.)
size = raveled_solution.shape
scaled_size = [(a*b) for (a,b) in zip(scale, size)]

x_gradient = np.zeros_like(raveled_solution)
x_gradient[:-1,:,:] = raveled_solution[:-1,:,:] - raveled_solution[1:,:,:]
x_gradient[-1,:,] = raveled_solution[-1,:,:]
x_cond = np.zeros_like(raveled_solution)
x_cond[:-1,:,:] = 2 / (1/cond_vol[:-1,:,:] + 1/cond_vol[1:,:,:])
x_cond[-1,:,:] = 0
x_speed = x_gradient * x_cond
q_x = (x_speed[0,:,:].sum() + x_speed[-1,:,:].sum())/2
k_x = (q_x * scaled_size[0]) / (scaled_size[1] * scaled_size[2]) # must be zero or close

y_gradient = np.zeros_like(raveled_solution)
y_gradient[:,:-1,:] = raveled_solution[:,:-1,:] - raveled_solution[:,1:,:]
y_gradient[:,-1,:] = raveled_solution[:,-1,:]
y_cond = np.zeros_like(raveled_solution)
y_cond[:,:-1,:] = 2 / (1/cond_vol[:,:-1,:] + 1/cond_vol[:,1:,:])
y_cond[:,-1,:] = 2 * cond_vol[:,-1,:]
y_speed = z_gradient * z_cond
q_y = (y_speed[:,0,:].sum() + y_speed[:,-1,:].sum())/2
k_y = (q_y * scaled_size[1]) / (scaled_size[0] * scaled_size[2])

z_gradient = np.zeros_like(raveled_solution)
z_gradient[:,:,:-1] = raveled_solution[:,:,:-1] - raveled_solution[:,:,1:]
z_gradient[:,:,-1] = raveled_solution[:,:,-1]
z_cond = np.zeros_like(raveled_solution)
z_cond[:,:,:-1] = 2 / (1/cond_vol[:,:,:-1] + 1/cond_vol[:,:,1:])
z_cond[:,:,-1] = 2 * cond_vol[:,:,-1]
z_speed = z_gradient * z_cond
q_z = (z_speed[:,:,0].sum() + z_speed[:,:,-1].sum())/2
k_z = (q_z * scaled_size[2]) / (scaled_size[0] * scaled_size[1])


In [31]:
print(k_x, k_y, k_z)

0.0 0.0 0.17000000476837157
